### Mount or unmount Gdrive

The drive by default is mounted in /content/drive/MyDrive

This cell is actually the only one that uses CoLab specific functions. You may as well use rclone mount or the like. That means that you could use the whole Notebook outside of CoLab, somewhere else, with some other storage as well ;-)


In [ ]:
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]

from google.colab import drive

# uncomment to suppress the debug output from drive.mount
drive.mount._DEBUG = False

if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  drive.flush_and_unmount()


### Setup And Update ComfyUI

We git clone from the official repo. Also, we may use a virtual environment inside the Google Drive so that we don't have to reinstall dependencies every time. This is controlled by the boolean variable ```USE_VENV```. 

#### About venv in CoLab

⚠️ Important Colab note : “Activating” a venv with source venv/bin/activate does NOT persist across cells in Colab. 👉 That’s why the correct approach is:
```
/path/to/venv/bin/python -m pip install ...
```
✅ Optional: Run ComfyUI inside the venv
```
!{VENV_PYTHON} main.py
```
#### Performance considerations

Running ComfyUI directly from the GDrive can be incredibly slow. Especially running venv from the drive can take a very long time (the pip modules are multiple GB in size...) This is why I introduced a third option: Bootstrapping from scratch (without Drive).

#### Bootstrapping from Scratch

My tests have shown that it may be faster to install ComfyUI from scratch and download everything from Github or Huggingface. Google Drive performance in my tests went down to about 100-200 MBit/sec while download from Huggingface consistently ran at 1-3 GBit/sec.

So if you set ```BOOTSTRAP_FROM_SCRATCH = True```, then ComfyUI will run on the ephemeral disk of the CoLab container. You will then have to download models for each workflow either by hand or by using custom nodes.

#### Downloading models

You typically need a bunch of (more or less large) model files from github, huggingface.co, civitai.com or the like. Whether you download these in a row and store them at home or in the Google Drive or whether you pull them each time you launch the platform, depends on disk space and network speed to these providers. But if you chose to download them on the fly each time you start a workflow, then I recommend using the ```INSTALL_CUSTOM_NODES``` boolean switch. If you set it to ```True```, then it will install the ComfyUI Downloader from https://github.com/romandev-codex/ComfyUI-Downloader.git - this is actually a server side download, meaning if you have a slow connection to the internet, then you can still download large models over the datacenter's connection.

In [ ]:
from pathlib import Path
import sys, os

# Set the drive path and update option
DRIVE_PATH = "/content/drive/MyDrive"  # @param {type:"string"}
USE_VENV = False  # @param {type:"boolean"}
BOOTSTRAP_FROM_SCRATCH = True  # @param {type:"boolean"}
INSTALL_CUSTOM_NODES = True  # @param {type:"boolean"}

if BOOTSTRAP_FROM_SCRATCH:
    print("Bootstrapping from scratch. Models will have to be downloaded for every workflow.")
    WORKSPACE = Path("/content/ComfyUI")
    VENV_PATH = Path("/content/venv.ComfyUI")
else:
    print("Running from the existing installation on the GDrive.")
    WORKSPACE = Path(DRIVE_PATH) / "ComfyUI"
    VENV_PATH = Path(DRIVE_PATH) / "venv.ComfyUI"

UPDATE_COMFY_UI = True  #@param {type:"boolean"}

# Check if the specified drive path exists
if (not BOOTSTRAP_FROM_SCRATCH) and (not os.path.isdir(DRIVE_PATH)):
    print(f"Error: Directory '{DRIVE_PATH}' does not exist. and BOOTSTRAP_FROM_SCRATCH is set to False. Please check and try again.")
    sys.exit(1)

# Create virtual environment if missing
if USE_VENV and not VENV_PATH.exists():
    print("Creating virtual environment...")
    !python3 -m venv --without-pip "{VENV_PATH}"  # Create venv without pip
    # Bootstrap pip manually
    !curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
    !{VENV_PATH}/bin/python get-pip.py
    !rm get-pip.py
elif USE_VENV:
    print("Virtual environment already exists.")
else:
    print("Using system Python.")

# Path to venv python
if USE_VENV:
    VENV_PYTHON = VENV_PATH / "bin" / "python"

%cd {DRIVE_PATH}

# Clone or update ComfyUI
if not WORKSPACE.exists():
    print("Cloning ComfyUI...")
    !git clone https://github.com/Comfy-Org/ComfyUI.git "{WORKSPACE}"

%cd {WORKSPACE}

# Update ComfyUI if the option is enabled
if UPDATE_COMFY_UI:
  !echo "ComfyUI exists, updating with git pull"
  !git pull

# Some remarks about the installation process:
# you might as well do the installation using comfy-cli
# !pip install comfy-cli
# !comfy install --workspace={WORKSPACE} --venv={VENV_PATH if USE_VENV else None}

# Upgrade pip
if USE_VENV:
    !{VENV_PYTHON} -m pip install --upgrade pip

# little helper function to install python requirements in a given subdirectory
def install_requirements(SubDir):
    %cd {SubDir}
    if USE_VENV:
        !{VENV_PYTHON} -m pip install -r requirements.txt
    else:
        !pip install -r requirements.txt

# Install the main requirements for ComfyUI
install_requirements(WORKSPACE)



# Install custom nodes if the option is enabled
if INSTALL_CUSTOM_NODES:
    print("Installing custom nodes")
    custom_nodes_path = WORKSPACE / 'custom_nodes'
    
    # clone the ComfUI Downloader node repo
    %cd {custom_nodes_path}
    print("Installing ComfyUI-Downloader custom node")
    !git clone https://github.com/romandev-codex/ComfyUI-Downloader.git
    print("Installing ComfyUI-Downloader custom node requirements")
    install_requirements(custom_nodes_path / "ComfyUI-Downloader")

    # clone the ComfyUI Manager node repo   
    %cd {custom_nodes_path}
    print("Installing ComfyUI-Manager custom node")
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git
    print("Installing ComfyUI-Manager custom node requirements")
    install_requirements(custom_nodes_path / "ComfyUI-Manager")

print("Setup complete. You can now run ComfyUI with:")
if USE_VENV:
    print(f"!{VENV_PYTHON} main.py --listen --port 8188")
else:
    print("!python main.py --listen --port 8188")



## Run ComfyUI

In [ ]:
import subprocess

%cd {WORKSPACE}

def has_gpu():
    try:
        subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True
        )
        return True
    except:
        return False

USE_GPU = has_gpu()
CPU_FLAG = "" if USE_GPU else "--cpu"

print("Using GPU" if USE_GPU else "Using CPU")

if USE_VENV:
    # run in venv
    !{VENV_PYTHON} main.py {CPU_FLAG} --listen --port 8188
else:
    # run with system python
    !python3 main.py {CPU_FLAG} --listen --port 8188
